##### *Reincidencia de sentenciados en prisiones del Perú*
# **Modelos implementados**
## **Integrantes**
* Renzo Dacillo
* Pierre Lavergne
* Luis Alberto Carrasco


## 1. Configuración y carga de datos sin fuga (Data Leakage)
Utilizamos el módulo `src.data_loader` conectado directamente al dataset procesado (`dataset04_onehot.csv`), garantizando que la imputación de medianas (`EDAD_SIN_MAMA`, `EDAD_SIN_PAPA`) y el escalado (`StandardScaler`) se ajusten únicamente sobre `X_train`.

In [ ]:
import sys
from pathlib import Path

# Agregar raiz del proyecto al path para importar src
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import src.data_loader as data_loader
import src.models as models
import src.evaluation as evaluation

sns.set_theme(style="whitegrid")

In [ ]:
X_train, X_test, y_train, y_test = data_loader.prepararDatos()
X_train_scaled, X_test_scaled = data_loader.escalarDatos(X_train, X_test)

pd.DataFrame({
    "Conjunto": ["Entrenamiento (Train)", "Prueba (Test)", "Total"],
    "Muestras": [len(X_train), len(X_test), len(X_train) + len(X_test)],
    "Features": [X_train.shape[1], X_test.shape[1], X_train.shape[1]],
    "Proporción Reincidentes": [y_train.mean(), y_test.mean(), pd.concat([y_train, y_test]).mean()]
})

## 2. Entrenamiento y comparación de modelos base
Dado el desbalanceo natural de la variable objetivo (~16.5% de reincidentes), configuramos los algoritmos mediante ponderación de clases (`class_weight='balanced'` y `scale_pos_weight`) en lugar de SMOTE en alta dimensión, evitando ruido sintético sobre variables categóricas One-Hot.

Modelos evaluados:
1. **Logistic Regression** (Baseline lineal sobre datos escalados)
2. **Random Forest** (Ensamble bagging)
3. **XGBoost** (Gradient Boosting)
4. **LightGBM** (Gradient Boosting basado en histogramas)

In [ ]:
modelos_base = models.entrenarModelosBase(X_train, X_train_scaled, y_train, random_state=42)
df_resultados_base, preds_base = evaluation.compararModelos(
    modelos_base, X_test, X_test_scaled, y_test, umbral=0.5
)

display(df_resultados_base.round(4))

In [ ]:
evaluation.graficarCurvasROC_PR(preds_base, y_test)
evaluation.graficarMatricesConfusion(preds_base, y_test)

## 3. Optimización de hiperparámetros (Validación Cruzada Estratificada)
Seleccionamos los modelos de Gradient Boosting (`LightGBM` y `XGBoost`) para ajustar sus hiperparámetros mediante `RandomizedSearchCV` (5-fold Stratified CV) optimizando **ROC-AUC**.

In [ ]:
lgbm_opt = models.optimizarModelo(
    "LightGBM", X_train, X_train_scaled, y_train, n_iter=15, cv=5, random_state=42
)

xgb_opt = models.optimizarModelo(
    "XGBoost", X_train, X_train_scaled, y_train, n_iter=15, cv=5, random_state=42
)

In [ ]:
modelos_finales = {
    "LogisticRegression (Base)": modelos_base["LogisticRegression"],
    "RandomForest (Base)": modelos_base["RandomForest"],
    "LightGBM (Optimizado)": lgbm_opt,
    "XGBoost (Optimizado)": xgb_opt
}

df_resultados_final, preds_final = evaluation.compararModelos(
    modelos_finales, X_test, X_test_scaled, y_test, umbral=0.5
)
display(df_resultados_final.round(4))

## 4. Ajuste de umbral de decisión y evaluación final del mejor modelo
Analizamos el mejor modelo según ROC-AUC y ajustamos el umbral de probabilidad para maximizar el equilibrio entre **Precision** y **Recall** (F1-Score) en la detección de internos reincidentes.

In [ ]:
mejor_nombre = df_resultados_final.index[0]
mejor_modelo_info = modelos_finales[mejor_nombre]
y_prob_mejor = preds_final[mejor_nombre]["y_prob"]

mejor_umbral, mejor_f1 = evaluation.encontrarMejorUmbral(y_test, y_prob_mejor)
print(f"Mejor modelo: {mejor_nombre}")
print(f"Umbral optimo (max F1): {mejor_umbral:.4f} -> F1-Score = {mejor_f1:.4f}\n")

metricas_umbral_opt = evaluation.evaluarModelo(
    mejor_modelo_info, X_test, X_test_scaled, y_test, umbral=mejor_umbral
)
evaluation.imprimirReporteDetallado(f"{mejor_nombre} (umbral={mejor_umbral:.2f})", y_test, metricas_umbral_opt["y_pred"])

evaluation.graficarMatricesConfusion(
    {
        f"{mejor_nombre} (umbral=0.50)": preds_final[mejor_nombre],
        f"{mejor_nombre} (umbral={mejor_umbral:.2f})": metricas_umbral_opt
    },
    y_test
)

## 5. Interpretabilidad: Factores principales asociados a la reincidencia
Extraemos las variables con mayor peso predictivo en el modelo ganador para contrastar los resultados estadísticos con el contexto criminológico y penitenciario.

In [ ]:
df_importancia = evaluation.graficarImportanciaVariables(
    mejor_modelo_info, X_train.columns, mejor_nombre, top_n=20
)

# Guardar el mejor modelo entrenado en models/
models.guardarModelo(mejor_modelo_info, "mejor_modelo_reincidencia.joblib")